In [1]:
import datetime

DATE = datetime.datetime.now()

# comment out line above and use line below if you want to fetch the data from a specific date
# Replace YYYY-MM-DD with your date in the same format
# DATE = datetime.datetime.fromisoformat("YYYY-MM-DD")

### Load api key from .env file into os.environ, then read env variable from there

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads variables from .env into os.environ

API_KEY = os.getenv("FIRM_API_KEY")
if not API_KEY:
    raise Exception("Missing FIRM_API_KEY. Set it in your environment or .env file.")

### Test api key

In [73]:
import requests

url = "https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY="
parameters = {"MAP_KEY": API_KEY}
response = requests.get(url, params=parameters)
if response.status_code == 200:
    print(f"Api request successful\n{'-' * 22}")

    # Convert the JSON response into a Python dictionary
    data = response.json()
    print(f"Current transactions: {data['current_transactions']}/5000")
else:
    raise Exception(
        f"An error occured with status code {response.status_code}: {response.reason}"
    )

Api request successful
----------------------
Current transactions: 0/5000


### Fetch data from the last 14 days

In [ ]:
import time
import requests
from pathlib import Path

date = DATE + datetime.timedelta(days=-1)
date_str = date.strftime("%Y-%m-%d")
data = []

for _i in range(0, 7):
    url = (
        "https://firms.modaps.eosdis.nasa.gov/api/area/csv/"
        + API_KEY
        + "/VIIRS_SNPP_NRT/world/2/"
        + date_str
    )
    date = date + datetime.timedelta(days=-2)
    date_str = date.strftime("%Y-%m-%d")
    response = requests.get(url)
    if response.status_code == 200:
        parsed_response = response.content.decode("utf-8")
        # if data is still empty, also append the column
        if not data:
            data.extend(parsed_response.split("\n"))
        else:
            data.extend(parsed_response.split("\n")[1:])
    else:
        raise Exception(
            f"An error occured with status code {response.status_code}: {response.reason}"
        )
    # api seems to behave well, but we sleep half second nonetheless
    time.sleep(0.5)
# use Path object so we don't have to worry about path separator differences
file_path = Path.cwd().parent.joinpath(
    "data", f"VIIRS_SNNP_NRT_world_14days_{DATE.strftime('%Y-%m-%d')}.csv"
)

with open(file_path, "w") as file:
    file.write("\n".join(data))


### Load csv into pandas dataframe, parse date and time into single timestamp, filter out unimportant columns and low confidence records.

In [1]:
import pandas as pd
from pathlib import Path
import datetime

date = "2026-05-02"

file_path = Path.cwd().parent.joinpath(
    "data", f"VIIRS_SNNP_NRT_world_14days_{date}.csv"
)

data = pd.read_csv(file_path)
# transform time in a string with actual utc mil time format
data["acq_time"] = data["acq_time"].apply(lambda x: f"{x:04d}")
data["timestamp"] = pd.to_datetime(
    data["acq_date"] + data["acq_time"], format="%Y-%m-%d%H%M", utc=True
)
data.drop(columns=["acq_date", "satellite", "instrument", "version", "acq_time"])
data = data[data["confidence"] != "l"]

data.info()
data.head(150000)


<class 'pandas.DataFrame'>
Index: 329115 entries, 0 to 379757
Data columns (total 15 columns):
 #   Column      Non-Null Count   Dtype              
---  ------      --------------   -----              
 0   latitude    329115 non-null  float64            
 1   longitude   329115 non-null  float64            
 2   bright_ti4  329115 non-null  float64            
 3   scan        329115 non-null  float64            
 4   track       329115 non-null  float64            
 5   acq_date    329115 non-null  str                
 6   acq_time    329115 non-null  str                
 7   satellite   329115 non-null  str                
 8   instrument  329115 non-null  str                
 9   confidence  329115 non-null  str                
 10  version     329115 non-null  str                
 11  bright_ti5  329115 non-null  float64            
 12  frp         329115 non-null  float64            
 13  daynight    329115 non-null  str                
 14  timestamp   329115 non-null  datetim

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,timestamp
0,32.33215,44.09279,306.72,0.71,0.75,2026-05-01,0001,N,VIIRS,n,2.0NRT,288.92,2.80,N,2026-05-01 00:01:00+00:00
1,32.88856,35.09304,296.63,0.44,0.46,2026-05-01,0001,N,VIIRS,n,2.0NRT,281.80,1.02,N,2026-05-01 00:01:00+00:00
2,33.15357,44.78751,302.25,0.73,0.76,2026-05-01,0001,N,VIIRS,n,2.0NRT,287.02,2.17,N,2026-05-01 00:01:00+00:00
3,33.15594,44.77987,316.28,0.73,0.76,2026-05-01,0001,N,VIIRS,n,2.0NRT,288.05,2.17,N,2026-05-01 00:01:00+00:00
4,33.15751,44.78342,342.93,0.73,0.76,2026-05-01,0001,N,VIIRS,n,2.0NRT,289.27,6.42,N,2026-05-01 00:01:00+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
174121,25.98571,95.88824,335.11,0.46,0.39,2026-04-23,0705,N,VIIRS,n,2.0NRT,302.80,3.98,D,2026-04-23 07:05:00+00:00
174122,25.98606,76.61487,353.22,0.78,0.78,2026-04-23,0705,N,VIIRS,n,2.0NRT,315.19,16.16,D,2026-04-23 07:05:00+00:00
174123,25.98611,95.88939,333.28,0.46,0.39,2026-04-23,0705,N,VIIRS,n,2.0NRT,301.98,2.48,D,2026-04-23 07:05:00+00:00
174124,25.99014,92.43753,330.28,0.38,0.36,2026-04-23,0705,N,VIIRS,n,2.0NRT,294.99,3.78,D,2026-04-23 07:05:00+00:00


### Load data into geodataframe

In [2]:
import geopandas as gpd

fire_data = gpd.GeoDataFrame(
    data, geometry=gpd.points_from_xy(data.longitude, data.latitude), crs="EPSG:4326"
)
fire_data.info()
fire_data.head()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 329115 entries, 0 to 379757
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype              
---  ------      --------------   -----              
 0   latitude    329115 non-null  float64            
 1   longitude   329115 non-null  float64            
 2   bright_ti4  329115 non-null  float64            
 3   scan        329115 non-null  float64            
 4   track       329115 non-null  float64            
 5   acq_date    329115 non-null  str                
 6   acq_time    329115 non-null  str                
 7   satellite   329115 non-null  str                
 8   instrument  329115 non-null  str                
 9   confidence  329115 non-null  str                
 10  version     329115 non-null  str                
 11  bright_ti5  329115 non-null  float64            
 12  frp         329115 non-null  float64            
 13  daynight    329115 non-null  str                
 14  timestamp   32911

,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,timestamp,geometry
0,32.33215,44.09279,306.72,0.71,0.75,2026-05-01,0001,N,VIIRS,n,2.0NRT,288.92,2.80,N,2026-05-01 00:01:00+00:00,POINT (44.09279 32.33215)
1,32.88856,35.09304,296.63,0.44,0.46,2026-05-01,0001,N,VIIRS,n,2.0NRT,281.80,1.02,N,2026-05-01 00:01:00+00:00,POINT (35.09304 32.88856)
2,33.15357,44.78751,302.25,0.73,0.76,2026-05-01,0001,N,VIIRS,n,2.0NRT,287.02,2.17,N,2026-05-01 00:01:00+00:00,POINT (44.78751 33.15357)
3,33.15594,44.77987,316.28,0.73,0.76,2026-05-01,0001,N,VIIRS,n,2.0NRT,288.05,2.17,N,2026-05-01 00:01:00+00:00,POINT (44.77987 33.15594)
4,33.15751,44.78342,342.93,0.73,0.76,2026-05-01,0001,N,VIIRS,n,2.0NRT,289.27,6.42,N,2026-05-01 00:01:00+00:00,POINT (44.78342 33.15751)
